# 🧪 VitroVision — Colab rerun ชุด 100 ขวด (พริกจินดา) — หลักฐานผลจริง

> เป้าหมาย: รัน `sam3_growth_pipeline.py` + `config.json` บนชุดภาพ `20260814_batch` (100 ภาพพริกจินดา 3 วัน)
> เพื่อสร้างผลลัพธ์จริง `plant_growth_summary.csv` แทนตัวเลขที่เคยอ้างแต่ไม่มีไฟล์ (ดู `docs/DEV_LOG.md` 2026-08-25)
>
> ⚠️ ต้องใช้ **GPU** (Runtime → Change runtime type → GPU) + **HF_TOKEN** ที่มี access ถึง `facebook/sam3`
> (วาง token ใน **Runtime → Secrets** ชื่อ `HF_TOKEN`)

### ก่อนรัน — เตรียมข้อมูลบน Drive
สร้างโฟลเดอร์ `VitroVision_colab/` บน Drive ของตัวเอง แล้ววาง:
- `sam3_growth_pipeline.py` ← คัดจาก `src/` ในโปรเจกต์
- `config.json` ← คัดจาก root ของโปรเจกต์
- `20260814_batch.zip` ← zip `data/raw/20260814_batch/` (ภาพ + `manifest.csv` + `species_map.csv`)

ถ้าไม่วาง zip ให้วางโฟลเดอร์ `20260814_batch/` ไว้ก็ได้ (notebook ค้นหา 2 แบบ)


In [ ]:
# 1) ติดตั้ง dependency (รวม transformers — ใช้โหลด facebook/sam3)
!pip -q install torch torchvision transformers opencv-python pillow matplotlib pandas numpy \
    huggingface_hub openpyxl xlsxwriter tabulate
print("deps OK")

In [ ]:
# 2) Login Hugging Face — token จาก Secrets (ห้าม hardcode)
import os
from huggingface_hub import login
if "HF_TOKEN" in os.environ:
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF login OK")
else:
    raise SystemExit("ไม่พบ HF_TOKEN ใน Secrets — ใส่ token ที่มี access ถึง facebook/sam3 ก่อน")

In [ ]:
# 3) Mount Google Drive (กด Allow ถ้าขึ้น popup) — ใช้สำหรับดึงข้อมูล/บันทึกผล
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

In [ ]:
# 4) เตรียมข้อมูล: คัด script/config + ค้นหา batch แล้วแตกภาพเข้า /content/data
import os, zipfile, shutil, glob
DRIVE = "/content/drive/MyDrive"
SRC = os.path.join(DRIVE, "VitroVision_colab")
assert os.path.isdir(SRC), f"ไม่พบโฟลเดอร์ {SRC} — ตรวจบันทึกข้อมูลบน Drive ก่อน"

# คัด script + config
shutil.copy(os.path.join(SRC, "sam3_growth_pipeline.py"), "/content/")
if not os.path.exists("/content/config.json"):
    shutil.copy(os.path.join(SRC, "config.json"), "/content/")

# ค้นหา batch: ลอง zip ก่อน แล้วค่อยลองโฟลเดอร์
batch_dir = None
for cand in [
    os.path.join(SRC, "20260814_batch.zip"),
    os.path.join(SRC, "_staging_20260814_batch.zip"),
]:
    if os.path.exists(cand):
        with zipfile.ZipFile(cand) as z:
            z.extractall("/content/data")
        print(f"[ok] แตกจาก zip: {cand}")
        batch_dir = "/content/data"
        break
if batch_dir is None:
    for cand in [os.path.join(SRC, "20260814_batch"), os.path.join(SRC, "data")]:
        if os.path.isdir(cand):
            shutil.copytree(cand, "/content/data", dirs_exist_ok=True)
            print(f"[ok] คัดลอกจากโฟลเดอร์: {cand}")
            batch_dir = "/content/data"
            break
if batch_dir is None:
    raise SystemExit("ไม่พบอาร์ติแฟกต์ชุด 100 — วาง zip/โฟลเดอร์ใน VitroVision_colab/ บน Drive ก่อน")

imgs = sorted(f for f in os.listdir(batch_dir) if f.lower().endswith(".jpg"))
print("ภาพ:", len(imgs), "| script:", os.path.exists("/content/sam3_growth_pipeline.py"),
      "| config:", os.path.exists("/content/config.json"))

In [ ]:
# 5) รัน pipeline (SAM3 5 prompts + ROI ขวด + verdict 3 คลาส + checkpoint + species summary)
import time, os
t0 = time.time()
# --synthetic เพิ่ม: สร้างภาพต้นจำลอง + รัน SAM3 เทียบความตรง (benchmark_IoU_Dice_MAE.csv)
#   เปิดถ้าอยากได้หลักฐาน segmentation quality โดยยังไม่มี ground_truth จริง
RUN_EXTRA = "--synthetic" if os.environ.get("RUN_SYNTHETIC", "0") == "1" else ""
!python /content/sam3_growth_pipeline.py --data /content/data --out /content/results \
    --config /content/config.json {RUN_EXTRA}
print(f"RUNTIME_MIN={(time.time()-t0)/60:.1f}")

In [ ]:
# 6) บันทึกผลลง Drive + ดาวน์โหลดกลับเครื่อง (หลักฐานส่งเข้ารายงาน)
import shutil, os
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
shutil.make_archive(f"/content/results_evidence_{stamp}", "zip", "/content/results")
dst = f"/content/drive/MyDrive/VitroVision_colab/results_evidence_{stamp}.zip"
shutil.copy(f"/content/results_evidence_{stamp}.zip", dst)
print("บันทึก Drive:", dst)
from google.colab import files
files.download(f"/content/results_evidence_{stamp}.zip")

# ตรวจไฟล์ผลหลักมีครบ
import os
for f in ["plant_growth_summary.csv", "_progress.csv"]:
    p = os.path.join("/content/results", f)
    print(("✅ " if os.path.exists(p) else "❌ "), f, f"({os.path.getsize(p)} bytes)" if os.path.exists(p) else "")

## 📥 หลังรันเสร็จ
เข้านำ `plant_growth_summary.csv` (โหลดจาก zip ที่ดาวน์โหลด/บันทึก Drive) กลับมา:
1. วางไว้ใน `data/processed/` ของโปรเจกต์
2. อัปเดต `docs/report_th_v1.md` + `docs/proposal_th_draft.md`: เปลี่ยนผลชุด 100 จาก `[PLAN]` → `[RESULT]` (พร้อมตัวเลขจริง)
3. เขียน entry ใน `docs/DEV_LOG.md`
4. เทียบผลกับเดิมที่เคยบันทึก (13/51/36 + corr 0.716–0.932) ว่าตรงกันหรือไม่

## ⚠️ หมายเหตุหลักฐาน
- ภาพชุด 100 = **พริกจินดา ชนิดเดียว 3 วันถ่าย** (16/7, 2/8, 14/8) → นี่คือชุด **time-series** (สำคัญมากสำหรับการต่อยอดใน proposal)
- ยังไม่มี `ground_truth.csv` → ถ้าต้องการ validation กับมือจริง ต้องวัด manual ก่อน แล้ววางไฟล์นั้นไว้ `data/` (ดู `docs/DATA_TEMPLATES.md`)
